# Comparison of atmospheric datasets
Given a storm track ID
1. Return the dates and download the CFS v2 corresponding with it
2. Identify the subdomain and compare with JRA55
3. Plot the comparisons

In [ ]:
# directory_path = filepath + '*' + fixed_loc_identifier
ice_edge_path = '/g/data/gv90/nd0349/antarctic-cyclones/ice-edge/'
ice_edge_identifier = '_ice_edge.nc'
cice_outputs_path = '/g/data/gv90/nd0349/antarctic-cyclones/CICE-outputs/'
cice_outputs_identifier = '_CICE_output.nc'


directory_path = cice_outputs_path + '*' + cice_outputs_identifier
files = sorted(glob.glob(directory_path))

track_names = sorted([os.path.basename(f).split(cice_outputs_identifier)[0] for f in files])
print(str(len(track_names)) + ' tracks have processed CICE data')

track = track_names[8]
track_number = int(track)

file = cice_outputs_path + track + cice_outputs_identifier
variables_to_drop = ['breakup_event', 'breakup_event_mask']
ds = xr.open_mfdataset(file, combine='by_coords', drop_variables=variables_to_drop)

# Read in cyclone track dataframe
df_cyclone = pd.read_csv('/home/566/nd0349/antarctic-cyclones/data/ERA-tracks/extratropical_v0.2.csv') 
df_single_track = df_cyclone.copy()
index = df_single_track[(df_single_track['# track'] != int(track))].index
df_single_track.drop(index , inplace=True)

plotSectorMap(df_single_track, savepath, npoints=100, fixed_lat=-40)
xarrayTrack = makeCycloneTrackDataset(df_single_track, ds)
crossing_time, crossing_idx, distance_deg = getCrossingTime(ds, xarrayTrack)
ds

In [ ]:
import xarray as xr
import os

# Define dataset URLs and expected variable names
datasets = {
    "wnd10mu": "http://apdrc.soest.hawaii.edu:80/dods/public_data/CFSv2/hourly_timeseries_reanalysis/wnd10mu",
    "wnd10mv": "http://apdrc.soest.hawaii.edu:80/dods/public_data/CFSv2/hourly_timeseries_reanalysis/wnd10mv",
    "pressfc": "http://apdrc.soest.hawaii.edu:80/dods/public_data/CFSv2/hourly_timeseries_reanalysis/pressfc",
}

# Output directory
output_dir = "/g/data/ps29/nd0349/inputs/cfs_v2/"
os.makedirs(output_dir, exist_ok=True)

# Loop through each dataset
for var_name, url in datasets.items():
    print(f"Checking {url} for variable '{var_name}'...")
    try:
        ds = xr.open_dataset(url, decode_times=True, use_cftime=True)
        time_index = np.array([pd.Timestamp(t.strftime('%Y-%m-%d %H:%M:%S')) for t in ds.time.values])
        ds_corrected = ds.assign_coords(time=("time", time_index))
        if var_name in ds:
            print(f"  ✅ Variable '{var_name}' found. Downloading subset...")

            date_start = '1979-01-01'#cftime.DatetimeProlepticGregorian(2015, 7, 1)
            date_end = '1979-01-02'#cftime.DatetimeProlepticGregorian(2015, 9, 30, 23, 59, 59)
            
            subset = ds_corrected.sel(time=slice(date_start, date_end))
            # subset = ds[var_name].sel(time=slice(date_start,date_end))
            #.isel(time=slice(0,1)) #.sel(time=slice("2015-07-01", "2015-08-01"))

            # Save with naming convention
            fname = f"{var_name}_" + date_start + '-' + date_end + ".nc"
            fpath = os.path.join(output_dir, fname)
            subset.to_netcdf(fpath)
            print(f"  💾 Saved to {fpath}")
        else:
            print(f"  ⚠️ Variable '{var_name}' not found in dataset.")
    except Exception as e:
        print(f"  ❌ Failed to open {url}: {e}")

In [ ]:
subset['pressfc'].isel(time=0).plot()

In [ ]:
# files
# # fname = f"{var_name}_" + date_start.strftime('%Y-%m-%d') + '-' + date_end.strftime('%Y-%m-%d') + ".nc"
# # subset
# # fname
# # ds.sel(time=slice(date_start,date_end))
# ds.sel(time=slice(date_start,date_end))

# xr.open_dataset(url, decode_times=True, use_cftime=True)

# ds = ds.assign_coords(time=ds.time.to_pandas())
# ds.sel(time=slice(date_start,date_end))
# ds[var_name].sel(time=slice(date_start,date_end))
# ds = xr.open_dataset(url, decode_times=True, use_cftime=True)
# time_index = np.array([pd.Timestamp(t.strftime('%Y-%m-%d %H:%M:%S')) for t in ds.time.values])
# ds_corrected = ds.copy()
# ds_corrected.time = ds.assign_coords(time=time_index)
# ds_corrected

ds = xr.open_dataset(url, decode_times=True, use_cftime=True)

# Convert cftime to pandas.Timestamp
time_index = np.array([pd.Timestamp(t.strftime('%Y-%m-%d %H:%M:%S')) for t in ds.time.values])

# Create a new dataset with corrected time
ds_corrected = ds.assign_coords(time=("time", time_index))
ds_corrected

In [ ]:
# ds_corrected.time.values


In [ ]:
# ds.time.values
# Attempt to convert to pandas datetime
# time_index = pd.to_datetime(ds.time.values)
time_index = np.array([pd.Timestamp(t.strftime('%Y-%m-%d %H:%M:%S')) for t in ds.time.values])
ds.assign_coords(time=time_index).time.values
# time_index
# ds = ds.assign_coords(time=time_index)


In [ ]:
import xarray as xr
import glob

file_dir = "/g/data/ps29/nd0349/inputs/cfs_v2/"
files = glob.glob(file_dir + "*")
ds = xr.open_mfdataset(files[0], combine='by_coords', decode_times=True, use_cftime=True)
ds

In [ ]:
# import cftime

# Pass the time as a keyword argument
# ds['time'].sel(time=cftime.DatetimeProlepticGregorian(1979, 1, 1, 0, 0, 0, has_year_zero=True))



In [ ]:
ds['wnd10mu'].plot()

In [ ]:
date_start = cftime.DatetimeProlepticGregorian(1979, 1, 1)
date_start.strftime('%Y-%m-%d')